# Setup

- presupposes call to extract_Erates_and_Pindex_from_MFMOD.py to create the csv containing the data
- this first cell 
    - reads csv into dataframe called **data**
    - creates country lists and mappings that are used later
    - creates function that filters data for sub-sample and country subset and normalises the exchange rates to starting period or first non-nan
    - creates some helper functions

In [ ]:
# Load data, extract exchange rates, and normalize them
import pandas as pd
import numpy as np
import altair as alt

# Load the CSV file into a pandas DataFrame
file_path = 'data/priceIndex_and_exchangeRate_series_2025_SP.csv'  # Replace with your CSV file path
data = pd.read_csv(file_path, index_col=0)

african_countries = [
    'DZA', 'AGO', 'BEN', 'BWA', 'BFA', 'BDI', 'CMR', 'CPV', 'CAF', 'TCD', 'COM', 'COG', 'CIV', 
    'COD', 'DJI', 'EGY', 'GNQ', 'ERI', 'SWZ', 'ETH', 'GAB', 'GMB', 'GHA', 'GIN', 'GNB', 'KEN', 
    'LBY', 'LSO', 'LBR', 'MDG', 'MWI', 'MLI', 'MRT', 'MUS', 'MAR', 'MOZ', 'NAM', 'NER', 'NGA', 
    'RWA', 'STP', 'SEN', 'SYC', 'SLE', 'SOM', 'ZAF', 'SSD', 'SDN', 'TZA', 'TGO', 'TUN', 'UGA', 
    'ZMB', 'ZWE'
]
CFA_countries = [
    'CMR', 'CAF', 'TCD', 'GNQ', 'GAB', 'COG', 'BEN', 'BFA', 'CIV', 'GNB', 'MLI', 'NER', 'SEN', 'TGO'
]
commodity_exporters = ['AGO', 'DZA', 'NGA', 'ZAF', 'EGY', 'MAR', 'TUN', 'CIV', 'GHA', 'BWA', 'NAM']
commodity_exportert = ['NGA', 'AGO', 'DZA', 'LBY', 'COG', 'GAB', 'TCD', 'SSD', 'GNQ', 'COD', 'ZMB', 'GIN']

# ISO3 code to full country name mapping
iso3_to_country_name = {
    'DZA': 'Algeria', 'AGO': 'Angola', 'BEN': 'Benin', 'BWA': 'Botswana', 'BFA': 'Burkina Faso',
    'BDI': 'Burundi', 'CMR': 'Cameroon', 'CPV': 'Cape Verde', 'CAF': 'Central African Republic',
    'TCD': 'Chad', 'COM': 'Comoros', 'COG': 'Congo', 'CIV': 'Ivory Coast', 'COD': 'Democratic Republic of the Congo',
    'DJI': 'Djibouti', 'EGY': 'Egypt', 'GNQ': 'Equatorial Guinea', 'ERI': 'Eritrea', 'SWZ': 'Eswatini',
    'ETH': 'Ethiopia', 'GAB': 'Gabon', 'GMB': 'Gambia', 'GHA': 'Ghana', 'GIN': 'Guinea', 'GNB': 'Guinea-Bissau',
    'KEN': 'Kenya', 'LSO': 'Lesotho', 'LBR': 'Liberia', 'MDG': 'Madagascar', 'MWI': 'Malawi', 'MLI': 'Mali',
    'MRT': 'Mauritania', 'MUS': 'Mauritius', 'MAR': 'Morocco', 'MOZ': 'Mozambique', 'NAM': 'Namibia',
    'NER': 'Niger', 'NGA': 'Nigeria', 'RWA': 'Rwanda', 'STP': 'Sao Tome and Principe', 'SEN': 'Senegal',
    'SYC': 'Seychelles', 'SLE': 'Sierra Leone', 'SOM': 'Somalia', 'ZAF': 'South Africa', 'SSD': 'South Sudan',
    'SDN': 'Sudan', 'TZA': 'Tanzania', 'TGO': 'Togo', 'TUN': 'Tunisia', 'UGA': 'Uganda', 'ZMB': 'Zambia',
    'ZWE': 'Zimbabwe', 'LBY': 'Libya'	
}

def extract_countries_and_sample(start, end = '2023-01-01', ma_window=5, countries=african_countries, df = data):
    # Filter out data before 2000 and non-African countries
    ma_window = 5 #years looking back in moving average
    start_date = pd.to_datetime(start)
    sd_minus_4 = start_date - pd.DateOffset(years=4) 
    df.index = pd.to_datetime(df.index)
    df = df[(df.index >= sd_minus_4) & (df.index <= end)]
    fX = pd.DataFrame(index=df.index)

    # Create a dictionary to store the new columns
    exchange_rates = {}
    first_non_nan_indices = {}
    # Populate the dictionary with the new columns
    for country in countries:
        panusatls_col = f"{country}PANUSATLS"
        price_level_col = f"{country}NECONPRVTXN"
        usa_price_level_col = "USANECONPRVTXN"
        
        if panusatls_col in df.columns and price_level_col in df.columns:
            exchange_rates[f"{country}_RER"] = (df[panusatls_col] * df[usa_price_level_col] / df[price_level_col])
            exchange_rates[f"{country}_NER"] = df[panusatls_col]
            exchange_rates[f"{country}_R2NER"] = (df[panusatls_col] / df[price_level_col])
            first_idx = exchange_rates[f"{country}_RER"].first_valid_index()
            first_non_nan_indices[country] = first_idx
        else:
            print(f"Missing columns for {country}: {panusatls_col}, {price_level_col}")

    # Add # Find the first non-NaN index for each country's _RER column
    #all the new columns to fX at once
    fX = pd.concat([fX, pd.DataFrame(exchange_rates)], axis=1)
    fX["USANECONPRVTXN"] = df["USANECONPRVTXN"]

    # normalise the rates to 100 at start date or first non nan
    normalised_rates = {}

    for country in countries:
        rer_col = f"{country}_RER"
        ner_col = f"{country}_NER"
        r2ner_col = f"{country}_R2NER"
        
        if rer_col in fX.columns and ner_col in fX.columns and r2ner_col in fX.columns and first_non_nan_indices[country] is not None:
            # Normalize the column and add moving average to r2ner    
            normalised_rates[f"{rer_col}_normalized"] = fX[rer_col] / fX[rer_col].loc[first_non_nan_indices[country]] * 100
            normalised_rates[f"{ner_col}_normalized"] = fX[ner_col] / fX[ner_col].loc[first_non_nan_indices[country]] * 100
            normalised_rates[f"{r2ner_col}_normalized"] = fX[r2ner_col] / fX[r2ner_col].loc[first_non_nan_indices[country]] * 100
            normalised_rates[f"US_price_level_normalized_to_{country}_base"] = fX["USANECONPRVTXN"] / fX["USANECONPRVTXN"].loc[first_non_nan_indices[country]] * 100
            normalised_rates[f"{country}_R2NER_MA"] = normalised_rates[f"{r2ner_col}_normalized"].rolling(window=ma_window).mean()
        else:
            print(f"Missing data for {country}: {rer_col} or {ner_col}")
    # Add all the new columns to fX at once
    fX = pd.concat([fX, pd.DataFrame(normalised_rates)], axis=1)

    return fX

def insert_rise(x, y, idx, y0=100.0):
    """
    Insert a point at the crossing with y0.
    """
    t = (y0 - y[idx]) / (y[idx + 1] - y[idx])  # 0 < t < 1
    x_cross = x[idx] + t * (x[idx + 1] - x[idx])
    y_cross = y0  # exactly on the threshold

    x_aug = np.insert(x, idx + 1, x_cross)
    y_aug = np.insert(y, idx + 1, y_cross)

    return pd.to_datetime(x_aug), np.asarray(y_aug)

def insert_return(x, y, idx, y0=100.0):
    """
    Insert a point at the crossing with y0.
    """
    t = (y0 - y[idx]) / (y[idx] - y[idx-1])  # 0 < t < 1
    x_cross = x[idx] + t * (x[idx] - x[idx-1])
    y_cross = y0  # exactly on the threshold

    x_aug = np.insert(x, idx, x_cross)
    y_aug = np.insert(y, idx, y_cross)

    return pd.to_datetime(x_aug), np.asarray(y_aug)

def extend_series(x, y, change_idx, y0=100.0):

    x_aug = []
    y_aug = []
    last = 0
    for i in change_idx:
        # original samples up to i
        x_aug.extend(x[last:i + 1])
        y_aug.extend(y[last:i + 1])

        # linear interpolation of the crossing with y0
        t = (y0 - y[i]) / (y[i + 1] - y[i])        # 0 < t < 1
        x_cross = x[i] + t * (x[i + 1] - x[i])

        x_aug.append(x_cross)
        y_aug.append(y0)                            # exactly on the threshold
        last = i + 1

    # append the remaining tail
    x_aug.extend(x[last:])
    y_aug.extend(y[last:])

    x_aug = pd.to_datetime(x_aug)
    y_aug = np.asarray(y_aug)

    # integrate ONLY the part above the line
    return x_aug, y_aug


FileNotFoundError: [Errno 2] No such file or directory: 'priceIndex_and_exchangeRate_series_2025_SP.csv'

In [ ]:
fX = extract_countries_and_sample(start='2007-01-01', end='2021-01-01')
country = "DZA"
r2ner_col = f"{country}_R2NER_normalized"
ma_col = f"{country}_R2NER_MA"
s = fX[r2ner_col]#.dropna()
values = s[s.index >= '2007-01-01'].values
times = s[s.index >= '2007-01-01'].index
ma_5year = fX.loc[fX.index >= '2007-01-01', ma_col].values
np.trapezoid(values/values[0] - 1) 

In [ ]:
np.sum(values/values[0] - 1) 
# Your current approach (correct for discrete payments)
dev = np.array([0.1, -0.05, 0.2])
t = np.array([1, 2, 3])
dfactors = (1.05) ** (-t)  # [0.952, 0.907, 0.864]
pv_sum = np.sum(dev * dfactors)  # 0.268

# Trapezoid approach (assumes continuous flows)
pv_trap = np.trapezoid(dev * dfactors, t)  # Different result
pv_sum, pv_trap

# Events

- this cell calculates all the depreciation events
    - within a specified time period (end date defaults to 2023)
    - for which there is a depreciation from the moving average baseline that exceeds a given threshold
    - that ends when another threshold it crossed on the way down
    - events are filtered to be non overlapping, keeping the most severe events
- it returns two important objects
    - **df_transitions**: a dataframe with an entry for each event capturing key statistics for futher analysis
    - **results_extended**: a dictionary country keys and event lists as values containing some key series for the subsequent plotting 

In [ ]:
# indentify the material depreciation events in real to nominal LCU:USD exchange rate
from scipy.signal import argrelextrema

# Define the thresholds for identifying peaks and troughs
start_date = "1994"
rise_threshold = 0.1  # 20% increase threshold
return_threshold = 0.1  # 15% return threshold

fX = extract_countries_and_sample(start = start_date)

results = []
results_extended = {}
for country in african_countries:
    r2ner_col = f"{country}_R2NER_normalized"
    ma_col = f"{country}_R2NER_MA"
    if r2ner_col not in fX.columns:
        continue
    s = fX[r2ner_col]#.dropna()
    if s.empty:
        continue
    values = s[s.index >= start_date].values
    times = s[s.index >= start_date].index
    ma_5year = fX.loc[fX.index >= start_date, ma_col].values


    trough_indices = argrelextrema(ma_5year, np.less)[0]
    first_non_nan_index = np.where(~np.isnan(ma_5year))[0][0]
    if ma_5year[first_non_nan_index] < ma_5year[first_non_nan_index+1]:
        trough_indices = np.insert(trough_indices, 0, first_non_nan_index)
    peak_indices = argrelextrema(values, np.greater)[0]
    if values[-1] > values[-2]:
        peak_indices = np.append(peak_indices, len(values) - 1)

    for t_idx in np.arange(len(times)): #trough_indices:
        possible_peaks = peak_indices[peak_indices > t_idx]
        if possible_peaks.size == 0:
            continue
        for p_idx in possible_peaks: # maybe loop over all peaks? or pick the highest peak?
            increase = values[p_idx] / ma_5year[t_idx] - 1.0
            if increase >= rise_threshold:
                threshold = ma_5year[t_idx] * (1. + return_threshold)
                fall_date = None
                fall_period = None
                area_under_curve = None
                gain_at_fall = None
                intermediate_troughs = 0
                for i in range(t_idx+1, p_idx):
                    if values[i] < threshold:
                        intermediate_troughs += 1
                if intermediate_troughs > 0:
                    continue # skip this event if there are intermediate troughs, as it is not a clean rise and fall
                
                # Use for-else construct: if no break occurs, use the last value.
                for idx in range(p_idx + 1, len(values)):
                    if values[idx] < threshold:
                        fall_date = times[idx]
                        return_period = times[idx] - times[t_idx]
                        gain_at_fall = (values[idx] - ma_5year[t_idx]) / ma_5year[t_idx] * 100
                        values_integrand = np.array(values, copy = True)
                        values_integrand[t_idx], values_integrand[idx] = np.maximum(values_integrand[t_idx],ma_5year[t_idx]), np.maximum(values_integrand[idx],ma_5year[t_idx]) # R2NER replaced with baseline at start and end date for clean integration
                        integrand = np.maximum(values_integrand/ma_5year[t_idx] - 1, 0.0)*100 # percentage deviation from baseline at event start date is the integrand
                        integration_period = (times[t_idx:idx+1]-times[t_idx]).days.astype('int64')/365
                        area_under_curve = np.trapezoid(integrand[t_idx:idx+1], integration_period) 
                        max_peak = p_idx
                        for i in peak_indices[peak_indices>p_idx]:
                            if values[i] > values[max_peak]:
                                max_peak = i
                        gain_at_peak = (values[max_peak] - ma_5year[t_idx]) / ma_5year[t_idx] * 100
                        break
                else:
                    # If fall condition never met, use the last time point
                    idx = len(values) - 1
                    fall_date = times[idx]
                    return_period = times[idx] - times[t_idx]
                    gain_at_fall = (values[idx] - ma_5year[t_idx]) / ma_5year[t_idx] * 100
                    values_integrand = np.array(values, copy = True)
                    values_integrand[t_idx], values_integrand[idx] = np.maximum(values_integrand[t_idx],ma_5year[t_idx]), np.maximum(values_integrand[idx],ma_5year[t_idx])
                    integrand = np.maximum(values_integrand/ma_5year[t_idx] - 1, 0.0)*100
                    integration_period = (times[t_idx:idx+1]-times[t_idx]).days.astype('int64')/365
                    area_under_curve = np.trapezoid(integrand[t_idx:idx+1], integration_period) 
                    max_peak = p_idx
                    for i in peak_indices[peak_indices>p_idx]:
                        if values[i] > values[max_peak]:
                            max_peak = i
                    gain_at_peak = (values[max_peak] - ma_5year[t_idx]) / ma_5year[t_idx] * 100

                results.append({
                    "Country": country,
                    "Event_Start_Date": times[t_idx],
                    "Event_Fall_Date": fall_date,
                    "Area_Under_Curve": area_under_curve,
                    "Years_to_return":return_period.days/365 if return_period is not None else None,
                    "Event_Peak_Date": times[max_peak],
                    "Percentage_Increase_to_Peak": gain_at_peak,
                    "Percentage_Increase_at_Return": gain_at_fall,
                    "Trough_Value": values[t_idx],
                    "MA_at_Trough": ma_5year[t_idx], 
                    "values_integrand": values_integrand,
                    "times": times}
                    )
                    
                if country not in results_extended:
                    results_extended[country] = []
                results_extended[country].append({
                "values_integrand": values_integrand,
                "times": times,
                "Area_Under_Curve": area_under_curve,
                "Event_Start_Date": times[t_idx],
                "Event_Fall_Date": fall_date,
                })

def filter_non_overlapping_events_df(df):
    filtered = []
    for country, group in df.groupby('Country'):
        # Sort by severity descending
        # print(group)
        group = group.sort_values('Area_Under_Curve', ascending=False)
        selected = []
        used_intervals = []
        for _, row in group.iterrows():
            start = row['Event_Start_Date']
            end = row['Event_Fall_Date']
            # Check for overlap with already selected intervals
            overlap = False
            for s, e in used_intervals:
                if not (end < s or start > e):
                    overlap = True
                    break
            if not overlap:
                selected.append(row)
                used_intervals.append((start, end))
        filtered.extend(selected)
    return pd.DataFrame(filtered) #.sort_values(['Country', 'Event_Start_Date'])


def filter_non_overlapping_events_dict(dict_data):
    filtered = {}
    for country, events in dict_data.items():
        # sort events in descending order by Area_Under_Curve
        sorted_events = sorted(events, key=lambda x: x["Area_Under_Curve"], reverse=True)
        selected = []
        used_intervals = []
        for event in sorted_events:
            start = event["Event_Start_Date"]
            end = event["Event_Fall_Date"]
            overlap = False
            for s, e in used_intervals:
                if not (end < s or start > e):
                    overlap = True
                    break
            if not overlap:
                selected.append(event)
                used_intervals.append((start, end))
        filtered[country] = selected
    return filtered

df_transitions = filter_non_overlapping_events_df(pd.DataFrame(results))
df_transitions.reset_index(drop=True, inplace=True)
results_extended = filter_non_overlapping_events_dict(results_extended)
df_transitions

## data_coverage

1. Calculate data coverage for each country by counting non-null observations in their exchange rate series
2. Compute incidence rates as events per year of available data
3. Calculate mean severity as the average Area Under Curve for each country's events

In [ ]:
# Calculate event incidence rate and mean severity for each country

# First, get the data coverage for each country (number of years with data)
data_coverage = {}
for country in african_countries:
    col = f"{country}_R2NER_normalized"
    if col in fX.columns:
        # Count non-null values and convert to years (assuming annual data)
        non_null_count = fX.loc[start_date:"2023", col].count()
        data_coverage[country] = non_null_count
    else:
        data_coverage[country] = 0

# Calculate event statistics by country
country_stats = []
for country in african_countries:
    country_events = df_transitions[df_transitions['Country'] == country]
    
    if country in data_coverage and data_coverage[country] > 0:
        # Event count
        event_count = len(country_events)
        
        # Data coverage in years
        years_coverage = data_coverage[country]
        
        # Incidence rate (events per year)
        incidence_rate = event_count / years_coverage if years_coverage > 0 else 0
        
        # Mean severity (mean area under curve)
        mean_severity = country_events['Area_Under_Curve'].mean() if event_count > 0 else 0
        
        # Add country name for plotting
        country_name = iso3_to_country_name.get(country, country)
        
        country_stats.append({
            'Country': country,
            'Country_Name': country_name,
            'Event_Count': event_count,
            'Years_Coverage': years_coverage,
            'Incidence_Rate': incidence_rate,
            'Mean_Severity': mean_severity
        })

# Convert to DataFrame
df_country_stats = pd.DataFrame(country_stats)

# Filter out countries with no events or no data
df_country_stats_filtered = df_country_stats[
    (df_country_stats['Event_Count'] > 0) & 
    (df_country_stats['Years_Coverage'] > 0)
].copy()

print("Country Statistics Summary:")
print(f"Countries with events: {len(df_country_stats_filtered)}")
print(f"Average incidence rate: {df_country_stats_filtered['Incidence_Rate'].mean():.3f} events/year")
print(f"Average mean severity: {df_country_stats_filtered['Mean_Severity'].mean():.1f}")

# Display top countries by incidence rate and severity
print("\nTop 5 countries by incidence rate:")
print(df_country_stats_filtered.nlargest(5, 'Incidence_Rate')[['Country_Name', 'Incidence_Rate', 'Mean_Severity']])

print("\nTop 5 countries by mean severity:")
print(df_country_stats_filtered.nlargest(5, 'Mean_Severity')[['Country_Name', 'Incidence_Rate', 'Mean_Severity']])

df_country_stats_filtered

1. Create a scatter plot showing the relationship between incidence rate and mean severity, with:
    - Point size/color indicating number of events
    - Tooltips showing detailed country information
    - Labels for countries with extreme values
2. Display histograms showing the distribution of both metrics

In [ ]:
# Create scatter plot of incidence rate vs mean severity
scatter_plot = alt.Chart(df_country_stats_filtered).mark_circle(size=60).encode(
    x=alt.X('Incidence_Rate:Q', 
            title='Event Incidence Rate (events per year)',
            scale=alt.Scale(zero=False)),
    y=alt.Y('Mean_Severity:Q', 
            title='Mean Severity (Area Under Curve)',
            scale=alt.Scale(zero=False)),
    color=alt.Color('Event_Count:Q', 
                    scale=alt.Scale(scheme='viridis'),
                    title='Number of Events'),
    tooltip=[
        alt.Tooltip('Country_Name:N', title='Country'),
        alt.Tooltip('Incidence_Rate:Q', title='Incidence Rate', format='.3f'),
        alt.Tooltip('Mean_Severity:Q', title='Mean Severity', format='.1f'),
        alt.Tooltip('Event_Count:Q', title='Event Count'),
        alt.Tooltip('Years_Coverage:Q', title='Years of Data', format='.1f')
    ]
).properties(
    title='Currency Depreciation: Event Incidence Rate vs Mean Severity',
    width=500,
    height=400
)

# Add country labels for extreme cases
text_labels = alt.Chart(df_country_stats_filtered).mark_text(
    align='left',
    baseline='middle',
    dx=7,
    fontSize=8
).encode(
    x='Incidence_Rate:Q',
    y='Mean_Severity:Q',
    text=alt.condition(
        # Label countries with high incidence rate OR high severity
        (alt.datum.Incidence_Rate > 0.15) | (alt.datum.Mean_Severity > 400),
        'Country:N',
        alt.value('')
    )
)

# Combine charts
final_chart = scatter_plot + text_labels

final_chart.display()

# Create additional histogram showing distribution of incidence rates
incidence_hist = alt.Chart(df_country_stats_filtered).mark_bar().encode(
    x=alt.X('Incidence_Rate:Q', bin=alt.Bin(maxbins=15), title='Incidence Rate (events/year)'),
    y=alt.Y('count():Q', title='Number of Countries')
).properties(
    title='Distribution of Event Incidence Rates',
    width=300,
    height=200
)

# Create histogram showing distribution of mean severity
severity_hist = alt.Chart(df_country_stats_filtered).mark_bar().encode(
    x=alt.X('Mean_Severity:Q', bin=alt.Bin(maxbins=15), title='Mean Severity'),
    y=alt.Y('count():Q', title='Number of Countries')
).properties(
    title='Distribution of Mean Severity',
    width=300,
    height=200
)

# Display histograms side by side
histograms = alt.hconcat(incidence_hist, severity_hist)
histograms.display()

# Pricing, by country and pooled

## Country-by-Country Analysis:
- Annual frequency (λ): Events per year based on actual data coverage
- Mean severity: Average loss amount when events occur
- Expected annual loss: λ × mean severity
- Risk measures: Standard deviation, coefficient of variation, min/max

In [ ]:
# Calculate expected annual loss for devaluation insurance product

# First, ensure we have the data coverage calculated
data_coverage = {}
for country in african_countries:
    col = f"{country}_R2NER_normalized"
    if col in fX.columns:
        # Count non-null values (assuming monthly data, convert to years)
        non_null_count = fX.loc[start_date:"2023", col].count()
        data_coverage[country] = non_null_count
    else:
        data_coverage[country] = 0

# Calculate actuarial metrics by country
actuarial_results = []

for country in african_countries:
    country_events = df_transitions[df_transitions['Country'] == country]
    
    if country in data_coverage and data_coverage[country] > 0:
        years_coverage = data_coverage[country]
        event_count = len(country_events)
        
        if event_count > 0:
            # Annual frequency (lambda)
            annual_frequency = event_count / years_coverage
            
            # Severity distribution statistics
            severities = country_events['Area_Under_Curve'].values
            mean_severity = np.mean(severities)
            std_severity = np.std(severities, ddof=1) if len(severities) > 1 else 0
            
            # Expected annual loss = frequency × mean severity
            expected_annual_loss = annual_frequency * mean_severity
            
            # Additional statistics for risk assessment
            max_severity = np.max(severities)
            min_severity = np.min(severities)
            
            # Calculate coefficient of variation for severity
            cv_severity = std_severity / mean_severity if mean_severity > 0 else 0
            
        else:
            annual_frequency = 0
            mean_severity = 0
            std_severity = 0
            expected_annual_loss = 0
            max_severity = 0
            min_severity = 0
            cv_severity = 0
        
        actuarial_results.append({
            'Country': country,
            'Country_Name': iso3_to_country_name.get(country, country),
            'Years_Coverage': years_coverage,
            'Event_Count': event_count,
            'Annual_Frequency': annual_frequency,
            'Mean_Severity': mean_severity,
            'Std_Severity': std_severity,
            'CV_Severity': cv_severity,
            'Min_Severity': min_severity,
            'Max_Severity': max_severity,
            'Expected_Annual_Loss': expected_annual_loss
        })

# Convert to DataFrame
df_actuarial = pd.DataFrame(actuarial_results)

# Filter to countries with data
df_actuarial_filtered = df_actuarial[df_actuarial['Years_Coverage'] > 0].copy()

print("=== COUNTRY-BY-COUNTRY ACTUARIAL ANALYSIS ===")
print(f"Countries analyzed: {len(df_actuarial_filtered)}")
print(f"Countries with events: {len(df_actuarial_filtered[df_actuarial_filtered['Event_Count'] > 0])}")

# Display top 10 countries by expected annual loss
print("\nTop 10 countries by Expected Annual Loss:")
top_countries = df_actuarial_filtered.nlargest(10, 'Expected_Annual_Loss')[
    ['Country_Name', 'Annual_Frequency', 'Mean_Severity', 'Expected_Annual_Loss']
].round(3)
print(top_countries)

# Summary statistics for countries with events
countries_with_events = df_actuarial_filtered[df_actuarial_filtered['Event_Count'] > 0]
print(f"\nSummary for {len(countries_with_events)} countries with events:")
print(f"Average Annual Frequency: {countries_with_events['Annual_Frequency'].mean():.4f}")
print(f"Average Mean Severity: {countries_with_events['Mean_Severity'].mean():.1f}")
print(f"Average Expected Annual Loss: {countries_with_events['Expected_Annual_Loss'].mean():.1f}")

df_actuarial_filtered

## Pooled Portfolio Analysis:
- Combines all country data for portfolio-level metrics
- Shows diversification benefits through reduced coefficient of variation
- Provides percentile analysis for capital adequacy

In [ ]:
# Calculate pooled portfolio analysis
print("\n=== POOLED PORTFOLIO ANALYSIS ===")

# Pool all events together
all_events = df_transitions.copy()
total_country_years = sum(data_coverage[country] for country in african_countries if data_coverage[country] > 0)
total_events = len(all_events)

if total_events > 0:
    # Pooled frequency
    pooled_annual_frequency = total_events / total_country_years
    
    # Pooled severity statistics
    all_severities = all_events['Area_Under_Curve'].values
    pooled_mean_severity = np.mean(all_severities)
    pooled_std_severity = np.std(all_severities, ddof=1)
    pooled_cv_severity = pooled_std_severity / pooled_mean_severity
    
    # Expected annual loss for pooled portfolio
    pooled_expected_annual_loss = pooled_annual_frequency * pooled_mean_severity
    
    # Risk measures
    pooled_max_severity = np.max(all_severities)
    pooled_min_severity = np.min(all_severities)
    
    # Calculate percentiles for severity distribution
    severity_percentiles = np.percentile(all_severities, [50, 75, 90, 95, 99])
    
    print(f"Total country-years of exposure: {total_country_years:.1f}")
    print(f"Total events observed: {total_events}")
    print(f"Pooled annual frequency: {pooled_annual_frequency:.4f} events per country-year")
    print(f"Pooled mean severity: {pooled_mean_severity:.1f}")
    print(f"Pooled std severity: {pooled_std_severity:.1f}")
    print(f"Coefficient of variation: {pooled_cv_severity:.2f}")
    print(f"Pooled expected annual loss: {pooled_expected_annual_loss:.1f} per country-year")
    
    print(f"\nSeverity Distribution Percentiles:")
    print(f"50th percentile: {severity_percentiles[0]:.1f}")
    print(f"75th percentile: {severity_percentiles[1]:.1f}")
    print(f"90th percentile: {severity_percentiles[2]:.1f}")
    print(f"95th percentile: {severity_percentiles[3]:.1f}")
    print(f"99th percentile: {severity_percentiles[4]:.1f}")
    print(f"Maximum: {pooled_max_severity:.1f}")

else:
    print("No events found in the dataset")

# Create comparison DataFrame
pooled_summary = pd.DataFrame({
    'Metric': ['Annual Frequency', 'Mean Severity', 'Expected Annual Loss', 'Std Severity', 'CV Severity'],
    'Pooled_Portfolio': [pooled_annual_frequency, pooled_mean_severity, pooled_expected_annual_loss, 
                        pooled_std_severity, pooled_cv_severity],
    'Country_Average': [countries_with_events['Annual_Frequency'].mean(),
                       countries_with_events['Mean_Severity'].mean(),
                       countries_with_events['Expected_Annual_Loss'].mean(),
                       countries_with_events['Std_Severity'].mean(),
                       countries_with_events['CV_Severity'].mean()]
})

print(f"\n=== POOLED vs COUNTRY AVERAGE COMPARISON ===")
print(pooled_summary.round(4))

## Key Actuarial Outputs:
- Expected annual loss per country and for pooled portfolio
- Risk loading recommendations based on volatility
- Capital requirements using extreme percentiles
- Diversification benefits from geographic pooling

In [ ]:
# Visualize the actuarial results
import altair as alt

# Filter to countries with events for plotting
countries_with_events = df_actuarial_filtered[df_actuarial_filtered['Event_Count'] > 0]

# Plot 1: Expected Annual Loss by country
loss_chart = alt.Chart(countries_with_events).mark_bar().encode(
    x=alt.X('Expected_Annual_Loss:Q', title='Expected Annual Loss'),
    y=alt.Y('Country_Name:N', sort='-x', title='Country'),
    color=alt.Color('Annual_Frequency:Q', scale=alt.Scale(scheme='viridis'), title='Annual Frequency'),
    tooltip=[
        'Country_Name:N',
        alt.Tooltip('Expected_Annual_Loss:Q', format='.1f'),
        alt.Tooltip('Annual_Frequency:Q', format='.4f'),
        alt.Tooltip('Mean_Severity:Q', format='.1f')
    ]
).properties(
    title='Expected Annual Loss by Country',
    width=600,
    height=400
)

# Plot 2: Frequency vs Severity with Expected Loss as size
scatter_chart = alt.Chart(countries_with_events).mark_circle().encode(
    x=alt.X('Annual_Frequency:Q', title='Annual Frequency (events/year)'),
    y=alt.Y('Mean_Severity:Q', title='Mean Severity (Area Under Curve)'),
    size=alt.Size('Expected_Annual_Loss:Q', title='Expected Annual Loss', scale=alt.Scale(range=[50, 500])),
    color=alt.Color('CV_Severity:Q', scale=alt.Scale(scheme='plasma'), title='Severity CV'),
    tooltip=[
        'Country_Name:N',
        alt.Tooltip('Annual_Frequency:Q', format='.4f'),
        alt.Tooltip('Mean_Severity:Q', format='.1f'),
        alt.Tooltip('Expected_Annual_Loss:Q', format='.1f'),
        alt.Tooltip('CV_Severity:Q', format='.2f')
    ]
).properties(
    title='Risk Profile: Frequency vs Severity (bubble size = Expected Loss)',
    width=500,
    height=400
)

# Display charts
loss_chart.display()
scatter_chart.display()

# Print summary for insurance pricing considerations
print(f"\n=== INSURANCE PRICING INSIGHTS ===")
print(f"1. Diversification benefit: Pooled portfolio has lower relative risk")
print(f"   - Individual country CV average: {countries_with_events['CV_Severity'].mean():.2f}")
print(f"   - Pooled portfolio CV: {pooled_cv_severity:.2f}")

print(f"\n2. Premium loading recommendations:")
print(f"   - Base premium (expected loss): {pooled_expected_annual_loss:.1f}")
print(f"   - Risk loading (e.g., 20% of std dev): {0.2 * pooled_std_severity:.1f}")
print(f"   - Total indicative premium: {pooled_expected_annual_loss + 0.2 * pooled_std_severity:.1f}")

print(f"\n3. Capital requirements (99th percentile): {severity_percentiles[4]:.1f}")

# Loan life analysis

In [ ]:
# Calculate 20-year rolling integrals for annual data
def calculate_20year_rolling_integrals(start_date="1994", end_date="2023"):
    """
    Calculate 20-year rolling integrals for all countries.
    Returns a list of dictionaries with country, start_date, end_date, and integral values.
    """
    results = []
    
    print("Calculating 20-year rolling integrals for annual data...")
    
    for country in african_countries:
        r2ner_col = f"{country}_R2NER_normalized"
        ma_col = f"{country}_R2NER_MA"
        
        if r2ner_col not in fX.columns or ma_col not in fX.columns:
            continue
            
        # Get country data for the full sample period
        country_data = fX.loc[start_date:end_date, [r2ner_col, ma_col]].dropna()
        
        if len(country_data) < 20:  # Need at least 20 years of annual data
            continue
            
        # Calculate all possible 20-year windows
        window_size = 20  # 20 years
        num_windows = len(country_data) - window_size + 1
        
        for start_idx in range(num_windows):
            end_idx = start_idx + window_size
            
            window_data = country_data.iloc[start_idx:end_idx]
            window_start = window_data.index[0]
            window_end = window_data.index[-1]
            
            # Get baseline (moving average at start of period)
            baseline = window_data[ma_col].iloc[0]
            
            if pd.isna(baseline) or baseline <= 0:
                continue
                
            # Calculate percentage deviation from baseline
            ratio = (window_data[r2ner_col] / baseline - 1) * 100
            
            # Calculate time in years from start (for annual data, this is just the index)
            t_years = np.arange(len(window_data))
            
            # Calculate integral (area under curve)
            valid_data = ratio.dropna()
            if len(valid_data) > 1:
                integral_value = np.trapezoid(valid_data, t_years[ratio.notna()])
                
                results.append({
                    'Country': country,
                    'Country_Name': iso3_to_country_name.get(country, country),
                    'Window_Start': window_start,
                    'Window_End': window_end,
                    'Integral_20Y': integral_value,
                    'Baseline_MA': baseline
                })
    
    print(f"Total results collected: {len(results)}")
    return results

# Calculate the rolling integrals
rolling_results = calculate_20year_rolling_integrals()

if rolling_results:
    df_20y_integrals = pd.DataFrame(rolling_results)
    print(f"\nSuccessfully created DataFrame with {len(df_20y_integrals)} rows")
    print(f"Countries with 20-year windows: {df_20y_integrals['Country'].nunique()}")
    
    # Summary statistics
    print(f"\n=== 20-YEAR INTEGRAL SUMMARY STATISTICS ===")
    print(f"Mean integral: {df_20y_integrals['Integral_20Y'].mean():.1f}%")
    print(f"Median integral: {df_20y_integrals['Integral_20Y'].median():.1f}%")
    print(f"Standard deviation: {df_20y_integrals['Integral_20Y'].std():.1f}%")
    print(f"Minimum integral: {df_20y_integrals['Integral_20Y'].min():.1f}%")
    print(f"Maximum integral: {df_20y_integrals['Integral_20Y'].max():.1f}%")
    
    # Calculate percentiles
    percentiles = np.percentile(df_20y_integrals['Integral_20Y'], [5, 10, 25, 50, 75, 90, 95, 99])
    print(f"\nPercentiles:")
    for i, p in enumerate([5, 10, 25, 50, 75, 90, 95, 99]):
        print(f"{p}th percentile: {percentiles[i]:.1f}%")
    
    # Count positive vs negative integrals
    positive_count = (df_20y_integrals['Integral_20Y'] > 0).sum()
    negative_count = (df_20y_integrals['Integral_20Y'] <= 0).sum()
    print(f"\nPositive integrals (net currency depreciation): {positive_count} ({positive_count/len(df_20y_integrals)*100:.1f}%)")
    print(f"Negative integrals (net currency appreciation): {negative_count} ({negative_count/len(df_20y_integrals)*100:.1f}%)")
    
    print(f"\nFirst few results:")
    print(df_20y_integrals.head())
    
else:
    print("No results found - checking data availability...")
    # Check data availability
    for country in african_countries[:10]:
        r2ner_col = f"{country}_R2NER_normalized"
        ma_col = f"{country}_R2NER_MA"
        
        if r2ner_col in fX.columns and ma_col in fX.columns:
            data_points = fX.loc["1994":"2023", [r2ner_col, ma_col]].dropna()
            print(f"{country}: {len(data_points)} data points (years: {data_points.index.min()} to {data_points.index.max()})")

In [ ]:
# Define typical project finance coverage ratios for comparison
typical_coverage_ratios = {
    'Conservative (AAA)': 200,  # 2.0x coverage ratio = 200% over baseline
    'Investment Grade': 150,    # 1.5x coverage ratio
    'Sub-Investment Grade': 100, # 1.0x coverage ratio
    'High Risk': 50            # 0.5x coverage ratio
}

print("=== COMPARISON WITH PROJECT FINANCE COVERAGE RATIOS ===")
print("\nTypical project life coverage ratios (% above baseline required for solvency):")
for rating, ratio in typical_coverage_ratios.items():
    count_above = (df_20y_integrals['Integral_20Y'] > ratio).sum()
    pct_above = count_above / len(df_20y_integrals) * 100
    print(f"{rating:20s}: {ratio:3d}% - Exceeded in {count_above:3d} cases ({pct_above:4.1f}%)")

# Calculate country-specific statistics
country_stats_20y = df_20y_integrals.groupby(['Country', 'Country_Name']).agg({
    'Integral_20Y': ['count', 'mean', 'std', 'min', 'max'],
}).round(1)

country_stats_20y.columns = ['Windows_Count', 'Mean_Integral', 'Std_Integral', 'Min_Integral', 'Max_Integral']
country_stats_20y = country_stats_20y.reset_index()

# Calculate risk metrics for each country
for rating, threshold in typical_coverage_ratios.items():
    col_name = f'Prob_Exceed_{rating.split()[0]}'
    country_risk = df_20y_integrals.groupby('Country')['Integral_20Y'].apply(
        lambda x: (x > threshold).mean() * 100
    ).reset_index()
    country_risk.columns = ['Country', col_name]
    country_stats_20y = country_stats_20y.merge(country_risk, on='Country', how='left')

print(f"\n=== TOP 10 RISKIEST COUNTRIES (by mean 20-year integral) ===")
top_risk = country_stats_20y.nlargest(10, 'Mean_Integral')[
    ['Country_Name', 'Windows_Count', 'Mean_Integral', 'Prob_Exceed_Conservative']
]
print(top_risk)

country_stats_20y

In [ ]:
# Create visualizations comparing 20-year integrals to coverage ratios

# Histogram of all 20-year integrals with coverage ratio thresholds
hist_chart = alt.Chart(df_20y_integrals).mark_bar(opacity=0.7).encode(
    x=alt.X('Integral_20Y:Q', 
            bin=alt.Bin(maxbins=50), 
            title='20-Year Cumulative Currency Impact (%)',
            scale=alt.Scale(domain=[-1000, 2000])),
    y=alt.Y('count():Q', title='Number of 20-Year Windows')
).properties(
    title='Distribution of 20-Year Currency Impact vs Project Finance Coverage Ratios',
    width=600,
    height=300
)

# Add vertical lines for coverage ratios
coverage_lines = []
colors = ['red', 'orange', 'yellow', 'green']
for i, (rating, threshold) in enumerate(typical_coverage_ratios.items()):
    line = alt.Chart(pd.DataFrame({'threshold': [threshold], 'rating': [rating]})).mark_rule(
        color=colors[i],
        strokeWidth=2,
        strokeDash=[5, 5]
    ).encode(
        x='threshold:Q',
        tooltip=['rating:N', 'threshold:Q']
    )
    coverage_lines.append(line)

# Combine histogram and threshold lines
combined_chart = alt.layer(hist_chart, *coverage_lines).resolve_scale(color='independent')

# Box plot by country (top 15 countries with most data)
top_countries = df_20y_integrals['Country'].value_counts().head(15).index
df_top_countries = df_20y_integrals[df_20y_integrals['Country'].isin(top_countries)]

box_chart = alt.Chart(df_top_countries).mark_boxplot(size=40).encode(
    x=alt.X('Country_Name:N', title='Country', sort='-y'),
    y=alt.Y('Integral_20Y:Q', 
            title='20-Year Cumulative Currency Impact (%)',
            scale=alt.Scale(domain=[-500, 1500])),
    tooltip=['Country_Name:N', 'Integral_20Y:Q']
).properties(
    title='20-Year Currency Impact Distribution by Country',
    width=800,
    height=400
)

# Display charts
combined_chart.display()
box_chart.display()

# Risk probability chart - FIXED VERSION
risk_data = []
for country in country_stats_20y['Country']:
    country_integrals = df_20y_integrals[df_20y_integrals['Country'] == country]['Integral_20Y']
    country_name = country_stats_20y[country_stats_20y['Country'] == country]['Country_Name'].iloc[0]
    
    for rating, threshold in typical_coverage_ratios.items():
        prob_exceed = (country_integrals > threshold).mean() * 100
        risk_data.append({
            'Country': country,
            'Country_Name': country_name,
            'Coverage_Ratio': rating,
            'Threshold': threshold,
            'Probability_Exceed': prob_exceed
        })

df_risk = pd.DataFrame(risk_data)

# Heatmap of exceedance probabilities - FIXED TOOLTIP
heatmap = alt.Chart(df_risk[df_risk['Country'].isin(top_countries[:10])]).mark_rect().encode(
    x=alt.X('Coverage_Ratio:N', title='Project Finance Rating Category'),
    y=alt.Y('Country_Name:N', title='Country'),
    color=alt.Color('Probability_Exceed:Q', 
                    scale=alt.Scale(scheme='reds', domain=[0, 100]),
                    title='Probability of\nExceeding Threshold (%)'),
    tooltip=[
        alt.Tooltip('Country_Name:N', title='Country'),
        alt.Tooltip('Coverage_Ratio:N', title='Rating Category'), 
        alt.Tooltip('Threshold:Q', title='Threshold (%)', format='.0f'),
        alt.Tooltip('Probability_Exceed:Q', title='Probability (%)', format='.1f')
    ]
).properties(
    title='Risk of Currency-Induced Project Insolvency by Country and Rating',
    width=400,
    height=300
)



# cost of risk associated with emergency liquidity and insolvency

In [ ]:
df_20y_integrals

In [ ]:
# Calculate emergency liquidity facility pricing based on 20-year integrals
print("=== EMERGENCY LIQUIDITY FACILITY PRICING ===")
print("Method: Direct calculation from 20-year integrals vs coverage ratios")

# For each coverage ratio, calculate the expected payout
facility_pricing = {}

for rating, coverage_threshold in typical_coverage_ratios.items():
    # Identify cases where emergency liquidity is needed
    liquidity_needed = df_20y_integrals['Integral_20Y'] > coverage_threshold
    
    # Calculate payouts (excess over coverage threshold)
    payouts = np.maximum(df_20y_integrals['Integral_20Y'] - coverage_threshold, 0)
    
    # Statistics
    prob_activation = liquidity_needed.mean()
    expected_payout = payouts.mean()
    expected_payout_given_activation = payouts[payouts > 0].mean() if (payouts > 0).any() else 0
    max_payout = payouts.max()
    
    # Risk measures
    payout_std = payouts.std()
    payout_95_percentile = np.percentile(payouts, 95)
    payout_99_percentile = np.percentile(payouts, 99)
    
    facility_pricing[rating] = {
        'Coverage_Threshold': coverage_threshold,
        'Prob_Activation': prob_activation,
        'Expected_Payout': expected_payout,
        'Expected_Payout_Given_Activation': expected_payout_given_activation,
        'Max_Payout': max_payout,
        'Payout_Std': payout_std,
        'Payout_95th_Percentile': payout_95_percentile,
        'Payout_99th_Percentile': payout_99_percentile,
        'Number_Activations': liquidity_needed.sum(),
        'Total_Windows': len(df_20y_integrals)
    }
    
    print(f"\n{rating} (Coverage Threshold: {coverage_threshold}%):")
    print(f"  Probability of activation: {prob_activation*100:.1f}%")
    print(f"  Expected payout (all cases): {expected_payout:.2f}%")
    print(f"  Expected payout given activation: {expected_payout_given_activation:.1f}%")
    print(f"  Maximum payout: {max_payout:.1f}%")
    print(f"  95th percentile payout: {payout_95_percentile:.1f}%")
    print(f"  99th percentile payout: {payout_99_percentile:.1f}%")
    print(f"  Cases requiring liquidity: {liquidity_needed.sum()}/{len(df_20y_integrals)}")

In [ ]:
# Calculate risk-adjusted pricing
print(f"\n=== RISK-ADJUSTED PRICING ===")

cost_of_capital = 0.12  # 12% cost of capital
target_roe = 0.15      # 15% target return on equity
risk_loading_factor = 1.5  # Additional loading for tail risk

pricing_summary = {}

for rating, metrics in facility_pricing.items():
    # Expected loss component
    expected_loss = metrics['Expected_Payout']
    
    # Capital requirement (use 99th percentile for economic capital)
    economic_capital = metrics['Payout_99th_Percentile']
    
    # Pricing components
    expected_loss_charge = expected_loss
    capital_cost_charge = economic_capital * cost_of_capital
    target_return_charge = economic_capital * (target_roe - cost_of_capital)
    risk_loading = expected_loss * (risk_loading_factor - 1)  # Additional loading for model uncertainty
    
    # Total pricing over 20 years
    total_facility_cost = expected_loss_charge + capital_cost_charge + target_return_charge + risk_loading
    
    # Annual equivalent pricing
    annual_facility_fee = total_facility_cost / 20
    
    pricing_summary[rating] = {
        'Expected_Loss_Charge': expected_loss_charge,
        'Capital_Cost_Charge': capital_cost_charge,
        'Target_Return_Charge': target_return_charge,
        'Risk_Loading': risk_loading,
        'Total_Facility_Cost': total_facility_cost,
        'Annual_Facility_Fee': annual_facility_fee,
        'Economic_Capital': economic_capital
    }
    
    print(f"\n{rating} Facility Pricing:")
    print(f"  Expected loss charge: {expected_loss_charge:.3f}%")
    print(f"  Capital cost charge: {capital_cost_charge:.3f}%")
    print(f"  Target return charge: {target_return_charge:.3f}%")
    print(f"  Risk loading: {risk_loading:.3f}%")
    print(f"  Total facility cost (20-year): {total_facility_cost:.3f}%")
    print(f"  Annual facility fee: {annual_facility_fee:.4f}%")
    print(f"  Economic capital requirement: {economic_capital:.1f}%")

In [ ]:
# Create summary tables and comparisons
print(f"\n=== FACILITY PRICING SUMMARY TABLE ===")
print(f"{'Rating':<20} {'Activation':<12} {'Expected':<12} {'Annual':<12} {'Economic':<12}")
print(f"{'':20} {'Prob (%)':<12} {'Payout (%)':<12} {'Fee (%)':<12} {'Capital (%)':<12}")
print("-" * 68)

for rating, pricing in pricing_summary.items():
    metrics = facility_pricing[rating]
    print(f"{rating:<20} {metrics['Prob_Activation']*100:>8.1f}{'':<4} {metrics['Expected_Payout']:>8.3f}{'':<4} {pricing['Annual_Facility_Fee']:>8.4f}{'':<4} {pricing['Economic_Capital']:>8.1f}")

# Compare to earlier event-based pricing
print(f"\n=== COMPARISON TO EVENT-BASED ANALYSIS ===")
if 'pooled_expected_annual_loss' in globals():
    print(f"Event-based annual expected loss: {pooled_expected_annual_loss:.3f}%")
    
    for rating in ['Investment Grade', 'Sub-Investment Grade']:
        if rating in pricing_summary:
            facility_fee = pricing_summary[rating]['Annual_Facility_Fee']
            ratio = facility_fee / pooled_expected_annual_loss if pooled_expected_annual_loss > 0 else float('inf')
            print(f"{rating} facility fee: {facility_fee:.4f}% ({ratio:.1f}x event-based)")

# Distribution analysis
print(f"\n=== PAYOUT DISTRIBUTION ANALYSIS ===")
investment_grade_threshold = typical_coverage_ratios['Investment Grade']
payouts = np.maximum(df_20y_integrals['Integral_20Y'] - investment_grade_threshold, 0)

print(f"Investment Grade (150% threshold) payout distribution:")
print(f"  Zero payout: {(payouts == 0).sum()}/{len(payouts)} cases ({(payouts == 0).mean()*100:.1f}%)")
print(f"  Mean positive payout: {payouts[payouts > 0].mean():.1f}% (when activated)")
print(f"  Median positive payout: {np.median(payouts[payouts > 0]):.1f}% (when activated)")
print(f"  Standard deviation: {payouts.std():.1f}%")
print(f"  Skewness: {pd.Series(payouts).skew():.2f}")
print(f"  Kurtosis: {pd.Series(payouts).kurtosis():.2f}")

# Key insights
print(f"\n=== KEY INSIGHTS ===")
print(f"1. Facility activation rates range from {min(m['Prob_Activation'] for m in facility_pricing.values())*100:.1f}% to {max(m['Prob_Activation'] for m in facility_pricing.values())*100:.1f}%")
print(f"2. Annual facility fees range from {min(p['Annual_Facility_Fee'] for p in pricing_summary.values())*1000:.1f} to {max(p['Annual_Facility_Fee'] for p in pricing_summary.values())*1000:.1f} basis points")
print(f"3. This pricing reflects the actual risk of currency-induced project distress")
print(f"4. Higher coverage ratios significantly reduce both activation probability and pricing")

In [ ]:
# Sensitivity analysis
print(f"\n=== SENSITIVITY ANALYSIS ===")

# Test different cost of capital assumptions
base_rating = 'Investment Grade'
if base_rating in facility_pricing:
    base_metrics = facility_pricing[base_rating]
    
    print(f"Sensitivity for {base_rating} facility:")
    print(f"{'Cost of Capital':<15} {'Annual Fee':<15} {'Total Cost':<15}")
    print("-" * 45)
    
    for coc in [0.08, 0.10, 0.12, 0.15, 0.18]:
        capital_charge = base_metrics['Payout_99th_Percentile'] * coc
        return_charge = base_metrics['Payout_99th_Percentile'] * max(0, target_roe - coc)
        risk_load = base_metrics['Expected_Payout'] * (risk_loading_factor - 1)
        
        total_cost = base_metrics['Expected_Payout'] + capital_charge + return_charge + risk_load
        annual_fee = total_cost / 20
        
        print(f"{coc*100:>8.0f}%{'':<7} {annual_fee:>10.4f}%{'':<4} {total_cost:>10.3f}%")

# Test different risk loading factors
print(f"\nRisk Loading Sensitivity:")
print(f"{'Risk Loading':<15} {'Annual Fee':<15} {'Total Cost':<15}")
print("-" * 45)

for risk_factor in [1.0, 1.25, 1.5, 2.0, 2.5]:
    capital_charge = base_metrics['Payout_99th_Percentile'] * cost_of_capital
    return_charge = base_metrics['Payout_99th_Percentile'] * (target_roe - cost_of_capital)
    risk_load = base_metrics['Expected_Payout'] * (risk_factor - 1)
    
    total_cost = base_metrics['Expected_Payout'] + capital_charge + return_charge + risk_load
    annual_fee = total_cost / 20
    
    print(f"{risk_factor:>8.2f}x{'':<6} {annual_fee:>10.4f}%{'':<4} {total_cost:>10.3f}%")

# Summary Statistics

- some basic analysis of **df_transitions**

In [ ]:
# Ensure Altair can handle large datasets
alt.data_transformers.disable_max_rows()

exclusion_list = [] # ["BFA", "CMR", "TCD", "DJI", "LSO", "SDN", "SWE"]
# df_transitions = pd.DataFrame(results)
df_transitions_to_chart = df_transitions[~df_transitions["Country"].isin(exclusion_list)]
# Histogram of length of drawdown period
chart_days = alt.Chart(df_transitions_to_chart).mark_bar().encode(
    alt.X("Years_to_return:Q", bin=alt.Bin(maxbins=30), title="Years"),
    alt.Y("count()", title="Count")
).properties(title="Time to Return", width=400, height=300)

# Histogram of Drawdown Area
chart_area = alt.Chart(df_transitions_to_chart).mark_bar().encode(
    alt.X("Area_Under_Curve:Q", bin=alt.Bin(maxbins=30), title="Percentage-Years over Baseline"),
    alt.Y("count():Q", title="Count")
).properties(title="Drawdown area", width=400, height=300)

histograms = alt.hconcat(*[chart_days, chart_area])
histograms.display()

# total number of events
total_events = df_transitions.shape[0]
print("Total number of events:", total_events)
# Count the number of events with no fall date
count_2023_end = df_transitions[df_transitions['Percentage_Increase_at_Return'] > 100*return_threshold].shape[0]
print("Event with no return date:", count_2023_end)
df_transitions[df_transitions.Years_to_return > 10]
print(df_transitions[df_transitions.Area_Under_Curve > 300])
# df_transitions.describe()
event_counts = df_transitions.groupby('Country').size().reset_index(name='Event_Count')
event_counts[event_counts.Event_Count > 1].sort_values('Event_Count', ascending=False)

# Plot time series with events

- creates separate plots for each country with the hybrid exchange rate, its moving average and the nominal exchange rate (on separate axis)
- highlights the areas for each event calculated above

In [ ]:
# plot the normalised exchange rates with the devaluation events and export to PDF
import math
keep_from_extended = ["times", "values_integrand"]
final_charts = []
three_countries = ['AGO']
for i, country in enumerate(african_countries): #african_countries

    # Define the column names for the current country
    r2ner_col_norm = f"{country}_R2NER_normalized"
    rer_col_norm   = f"{country}_RER_normalized"
    ner_col_norm   = f"{country}_NER_normalized"
    ner_col   = f"{country}_NER"
    ma_col = f"{country}_R2NER_MA"

    legend_setting = alt.Legend(title="Exchange rates", 
                                orient= "bottom", 
                                labelExpr="substring(datum.label, 4, length(datum.label)) === 'NER' ? 'Nominal (right axis)' : substring(datum.label, 10, length(datum.label)) === 'MA' ? 'Moving average' : 'RealLCY to NominalUSD '") if i%2 == 0 else None
    
    # Ensure the necessary columns exist in fX for this country.
    if all(col in fX.columns for col in [r2ner_col_norm, ner_col]):
        # Prepare data in long format
        data = fX[fX.index >= start_date].reset_index()[['index', ner_col, r2ner_col_norm, ma_col]]
        data_long = data.melt(
            id_vars=['index'],
            value_vars=[ner_col, r2ner_col_norm,ma_col],
            var_name='Metric',
            value_name='Normalized_Value'
        )
        
        # Prepare background intervals from transitions for the current country.
        trans = df_transitions[df_transitions['Country'] == country]
        end_date = fX.index.max()
        red_backgrounds = []
        blue_annotations = []
        j=-1
        for _, row in trans.iterrows():
            j=j+1
            # Shade the area below the R2NER curve between Event_start_date and Event_Peak_Date
            event_extended = results_extended[country][j]
            extended_series = {k: event_extended[k] for k in keep_from_extended}
            data_extended = pd.DataFrame(extended_series)
            red_shade_data = data_extended[(data_extended["times"] >= row["Event_Start_Date"]) &
                                           (data_extended["times"] <= row["Event_Fall_Date"])].assign(MA_at_Trough=row['MA_at_Trough'])
                                            # data[(data['index'] >= row['Event_start_date']) &
                                            #                   (data['index'] <= row['Event_Peak_Date'])].assign(MA_at_Trough=row['MA_at_Trough'])
            red_rect = alt.Chart(red_shade_data).mark_area(
                opacity=0.2,
                color='red'
            ).encode(
                x=alt.X("times:T"),
                y=alt.Y("values_integrand:Q"),
                y2=alt.Y2("MA_at_Trough:Q")
            )
            red_backgrounds.append(red_rect)

            # Add blue text annotation under the trough_value with the value of Area_Under_Curve
            if pd.notnull(row['Area_Under_Curve']):
                x_pos_blue = row['Event_Start_Date'] + (row['Event_Fall_Date'] - row['Event_Start_Date']) * 4/7
                blue_text_df = pd.DataFrame({'x': [x_pos_blue], 'y': [row['Trough_Value']+3], 'text': [f"{row['Area_Under_Curve']:.0f}%"]})
                blue_text = alt.Chart(blue_text_df).mark_text(fontSize=12, dy=5, color='red').encode(
                    x=alt.X('x:T'),
                    y=alt.Y('y:Q'),
                    text=alt.Text('text:N')
                )
                blue_annotations.append(blue_text)

        # Create two charts: one for metrics other than ner_col_norm (left y-axis) 
        # and one exclusively for ner_col_norm (right y-axis)

        left_data = data_long[data_long['Metric'] != ner_col]
        right_data = data_long[data_long['Metric'] == ner_col]
        
        left_chart = alt.layer(
            alt.Chart(left_data).mark_line(point=True).encode(
            x=alt.X("index:T", title="Year", axis=alt.Axis(format="%Y")),
            y=alt.Y("Normalized_Value:Q", title="Left Axis Value"),
            color=alt.Color("Metric:N"),
            tooltip=["index:T", "Metric", "Normalized_Value"]
            ),
            *red_backgrounds,
            *blue_annotations
        )
        
        right_chart = alt.Chart(right_data).mark_line(point=True).encode(
            x=alt.X("index:T", axis=alt.Axis(format="%Y")),
            y=alt.Y("Normalized_Value:Q", title="Right Axis Value", axis=alt.Axis(orient='right')),
            color=alt.Color("Metric:N", legend=legend_setting),
            tooltip=["index:T", "Metric", "Normalized_Value"]
        )
        
        # Layer the two charts and set independent y-scales for dual axes.
        base = alt.layer(left_chart, right_chart).resolve_scale(
            y='independent'
        )
    
        
        # Combine background intervals, annotations, and base chart.
        # Disable axes for the annotation layers to prevent duplicate y-axis labels.
        final_chart = alt.layer(base).properties(
            title=f"Real depreciation events in {iso3_to_country_name[country]}",
            width=400,
            height=400
        )
        # final_chart.display()
        # print(df_transitions[df_transitions.Country == country][['Country', 'Area_Under_Curve']])

        final_charts.append(final_chart)

# final_charts is assumed to be a list where each iteration appended the generated final_chart.
# For example, modify the loop in cell 2 to include:
#     final_charts.append(final_chart)
#
# This snippet arranges the charts into rows of 2 and exports them as a single PDF.
def chart_list_to_rows_of_two(chart_list):
    rows = math.ceil(len(chart_list) / 2)
    chart_rows = []
    for i in range(rows):
        charts_in_row = chart_list[i*2 : i*2 + 2]
        if len(charts_in_row) == 2:
            # Resolve the color scales independently so that each chart retains its own color encoding.
            row = alt.hconcat(*charts_in_row).resolve_scale(color='independent')
        else:
            row = charts_in_row[0]
        chart_rows.append(row)

    final_layout = alt.vconcat(*chart_rows)
    return final_layout

final_layout = chart_list_to_rows_of_two(final_charts)
# final_layout.save("chartsFrom5Yma_as_baseline.pdf")
final_layout

# Full integrals

- compute integrals over longer periods
    - 1994-2008
    - 2009-2023
    - 1994-2023

In [ ]:
# import altair as alt
def compute_integrals(start_date: str, end_date: str) -> pd.DataFrame:
    integrals = {}
    name = "area" + start_date[:4] + "to" + end_date[:4]
    for country in african_countries:
        r2ner_col = f"{country}_R2NER_normalized"
        ma_col = f"{country}_R2NER_MA"
        # Proceed only if both columns exist in fX
        if r2ner_col in fX.columns and ma_col in fX.columns:
            df_sub = fX.loc[start_date:end_date]
            if df_sub.empty:
                integrals[country] = np.nan
                continue
            # Get baseline from the start_date (or the first available date)
            if start_date in df_sub.index.astype(str):
                baseline = df_sub.loc[start_date, ma_col]
            else:
                baseline = df_sub[ma_col].iloc[0]
            ratio = (df_sub[r2ner_col] / baseline - 1)*100
            t_years = (df_sub.index - pd.Timestamp(start_date)).days / 365.25
            valid = ratio.notna() & np.isfinite(ratio)
            if valid.all():
                integral_value = np.trapezoid(ratio[valid], t_years[valid])
                integrals[country] = integral_value
            else:
                integrals[country] = np.nan
        else:
            integrals[country] = np.nan
    return pd.DataFrame({
        "Country": list(integrals.keys()),
        name: list(integrals.values())
    })

area_1994_2008 = compute_integrals("1994-01-01", "2008-01-01")
# Merge the computed integral dataframes into one combined dataframe using 'Country' as key
combined_area = (area_1994_2008
                 .merge(area_2009_2023, on="Country", how="outer")
                 .merge(area_1994_2023, on="Country", how="outer"))
area_2009_2023 = compute_integrals("2009-01-01", "2023-01-01")
area_1994_2023 = compute_integrals("1994-01-01", "2023-01-01")
combined_area = (area_1994_2008
                 .merge(area_2009_2023, on="Country", how="outer")
                 .merge(area_1994_2023, on="Country", how="outer"))

chart_integral_94to08 = alt.Chart(area_1994_2008).mark_bar().encode(
    alt.X("area1994to2008:Q", bin=alt.Bin(maxbins=30), title="Percent-Years"),
    alt.Y("count()", title="Count")
).properties(title="Total area under the curve from 1994 to 2009", width=400, height=300)

chart_integral_09to23 = alt.Chart(area_2009_2023).mark_bar().encode(
    alt.X("area2009to2023:Q", bin=alt.Bin(maxbins=30), title="Percent-Years"),
    alt.Y("count()", title="Count") 
).properties(title="Total area under the curve from 2009 to 2023", width=400, height=300)

chart_integral_94to23 = alt.Chart(area_1994_2023).transform_calculate(
    # For values above 5000 subtract 4000 to “compress” the empty gap between 1000 and 5000.
    x_transformed="datum.area1994to2023 > 4000 ? datum.area1994to2023 - 3000 : datum.area1994to2023",
    is_high="(datum.area1994to2023 > 4000) ? 'Above 4000' : 'Below 4000'"
).mark_bar().encode(
    alt.X("x_transformed:Q", 
          bin=alt.Bin(maxbins=30), 
          title="Percent-Years",
          # For ticks above 1000 (transformed values), add 4000 back so that the label shows the original value.
          axis=alt.Axis(labelExpr="datum.value < 1000 ? datum.value : datum.value + 3000")
    ),
    alt.Y("count()", title="Count"),
    color=alt.Color("is_high:N", scale=alt.Scale(domain=["Below 4000", "Above 4000"], range=["steelblue", "red"]), legend=alt.Legend(title="Original Value"))
).properties(
    title="Total area under the curve from 1994 to 2023", 
    width=400, 
    height=300
)

subsamples_combined_series = pd.concat([pd.Series(area_1994_2008.area1994to2008),pd.Series(area_2009_2023.area2009to2023)], ignore_index=True)
subsamples_combined_dict = {"areas": subsamples_combined_series}
subsamples_combined_df = pd.DataFrame(subsamples_combined_dict)

chart_integral_subsamples_combined = alt.Chart(subsamples_combined_df).mark_bar().encode(
    alt.X("areas:Q", bin=alt.Bin(maxbins=30), title="Percent-Years"),
    alt.Y("count()", title="Count")
).properties(title="Cumulative devaluation 94-08 & 09-23 as distinct draws", width=400, height=300)

sub_sample_histograms = alt.hconcat(*[chart_integral_94to08, chart_integral_09to23])
full_sample_histograms= alt.hconcat(*[chart_integral_subsamples_combined, chart_integral_94to23])
sub_sample_histograms.display()
full_sample_histograms.display()

area_1994_2023[area_1994_2023.area1994to2023> 4000]
combined_area[(combined_area.area1994to2008 > 300) | (combined_area.area2009to2023 > 300)]

In [ ]:
import altair as alt
from vega_datasets import data

# Prepare the TopoJSON and your data for joining
countries = alt.topo_feature(data.world_110m.url, 'countries')

# Prepare a DataFrame with ISO3 and Area_Under_Curve (or any variable)
# Create a base DataFrame with all African countries and a default area_under_curve value of 0
base_df = pd.DataFrame({'Country': african_countries, 'Area_Under_Curve': 0, 'Years_to_return': 0})

# From df_transitions, select only the transitions for African countries and, for each country, get the maximum Area_Under_Curve
transitions_max = (
    df_transitions[df_transitions['Country'].isin(african_countries)]
    .groupby('Country', as_index=False)['Area_Under_Curve']
    .max()
)
length_max = (
    df_transitions[df_transitions['Country'].isin(african_countries)]
    .groupby("Country", as_index=False)['Years_to_return'].max()
)

transitions_max = pd.merge(transitions_max, length_max, on='Country', how='outer')

# Merge the base DataFrame with the maximum transition values; countries without transitions remain 0
africa_choropleth = pd.merge(base_df, transitions_max, on='Country', how='left', suffixes=('', '_trans'))
africa_choropleth['Area_Under_Curve'] = africa_choropleth['Area_Under_Curve_trans'].fillna(africa_choropleth['Area_Under_Curve'])
# africa_choropleth = africa_choropleth[['Country', 'Area_Under_Curve', 'Years_to_return']]

# Create a mapping from ISO 3166-1 alpha-3 to ISO 3166-1 numeric codes
iso3166_alpha3_to_num = {
    'DZA': 12, 'AGO': 24, 'BEN': 204, 'BWA': 72, 'BFA': 854, 'BDI': 108, 'CMR': 120, 'CPV': 132,
    'CAF': 140, 'TCD': 148, 'COM': 174, 'COG': 178, 'CIV': 384, 'COD': 180, 'DJI': 262, 'EGY': 818,
    'GNQ': 226, 'ERI': 232, 'SWZ': 748, 'ETH': 231, 'GAB': 266, 'GMB': 270, 'GHA': 288, 'GIN': 324,
    'GNB': 624, 'KEN': 404, 'LSO': 426, 'LBR': 430, 'MDG': 450, 'MWI': 454, 'MLI': 466, 'MRT': 478,
    'MUS': 480, 'MAR': 504, 'MOZ': 508, 'NAM': 516, 'NER': 562, 'NGA': 566, 'RWA': 646, 'STP': 678,
    'SEN': 686, 'SYC': 690, 'SLE': 694, 'SOM': 706, 'ZAF': 710, 'SSD': 728, 'SDN': 729, 'TZA': 834,
    'TGO': 768, 'TUN': 788, 'UGA': 800, 'ZMB': 894, 'ZWE': 716, 'LBY': 434	
}

# Convert the iso3166_alpha3_to_num mapping to a DataFrame and merge with africa_choropleth
iso_mapping_df = pd.DataFrame(list(iso3166_alpha3_to_num.items()), columns=['Country', 'ISO_3166_1_num'])
africa_choropleth = africa_choropleth.merge(iso_mapping_df, on='Country', how='left')

# Join the full country names using the iso3_to_country_name dictionary
africa_choropleth['Country_Name'] = africa_choropleth['Country'].map(iso3_to_country_name)

# List of numeric codes for filtering (used here only for lookup)
africa_numeric_ids = list(iso3166_alpha3_to_num.values())

# Create the map: all countries are plotted, and those missing Area_Under_Curve are filled white.
choropleth = alt.Chart(countries).mark_geoshape(
    stroke='green'  # add a border to separate countries from the sea
).encode(
    # If a country has a value then show the color, otherwise fill white.
    color=alt.condition(
        "datum.Area_Under_Curve != null",
        alt.Color('Area_Under_Curve:Q', scale=alt.Scale(scheme='blues'), title='Area Under Curve'),
        alt.value("white")
    ),
    tooltip=[
        alt.Tooltip('Country_Name:N', title='Country'),
        alt.Tooltip('Area_Under_Curve:Q', title='Area Under Curve')
    ]
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(africa_choropleth, 'ISO_3166_1_num', ['Area_Under_Curve', 'Country_Name'])
).project(
    type="mercator",
    center=[17, 0],   # Center on Africa
    scale=250
).properties(
    width=500,
    height=500,
    title='Choropleth of Area Under Curve in Africa'
)

choropleth.display()

In [ ]:
countries = alt.topo_feature(data.world_110m.url, 'countries')

# Prepare a DataFrame with ISO3 and Area_Under_Curve (or any variable)
# Create a base DataFrame with all African countries and a default area_under_curve value of 0
base_df = pd.DataFrame({'Country': african_countries, 'Area_Under_Curve': 0, 'Years_to_return': 0})

# From df_transitions, select only the transitions for African countries and, for each country, get the maximum Area_Under_Curve
transitions_max = (
    df_transitions[df_transitions['Country'].isin(african_countries)]
    .groupby('Country', as_index=False)['Area_Under_Curve']
    .max()
)
length_max = (
    df_transitions[df_transitions['Country'].isin(african_countries)]
    .groupby("Country", as_index=False)['Years_to_return'].max()
)

transitions_max = pd.merge(transitions_max, length_max, on='Country', how='outer')

# Merge the base DataFrame with the maximum transition values; countries without transitions remain 0
africa_choropleth = pd.merge(base_df, transitions_max, on='Country', how='left', suffixes=('', '_trans'))
africa_choropleth['Area_Under_Curve'] = africa_choropleth['Area_Under_Curve_trans'].fillna(africa_choropleth['Area_Under_Curve'])
# africa_choropleth = africa_choropleth[['Country', 'Area_Under_Curve', 'Years_to_return']]

In [ ]:
africa_choropleth = pd.merge(base_df, transitions_max, on='Country', how='left', suffixes=('', '_trans'))
africa_choropleth

# below are old cells as reference for boilerplate code

In [ ]:
# Loop through each country and plot the specified columns within the date range using Altair
long_term_plots = []
for country in african_countries:
    col_r2ner_norm = f"{country}_R2NER_normalized"
    col_r2ner_ma   = f"{country}_R2NER_MA"
    col_rer_norm   = f"{country}_RER_normalized"
    col_ner_norm   = f"{country}_NER_normalized"
    
    # Check which of the desired columns exist in fX
    cols = [col for col in [col_rer_norm, col_r2ner_norm, col_r2ner_ma, col_ner_norm] if col in fX.columns]
    if not cols:
        print(f"None of the columns exist for {country}.")
        continue

    # Filter the data for the period from 1990 to 2023
    data_to_plot = fX.loc["1990":"2023", cols]
    if data_to_plot.empty:
        print(f"No data available from 1990 to 2023 for {country}.")
        continue

    # Reset index and convert the date index into a column for Altair
    data_to_plot = data_to_plot.reset_index().rename(columns={'index': 'date'})
    # Reshape the DataFrame into long format
    data_long = data_to_plot.melt(id_vars=["date"], value_vars=cols, 
                                  var_name="Metric", value_name="Normalized_Value")

    # Separate data: left axis for r2ner and its MA; right axis for ner metric
    left_data = data_long[~(data_long.Metric == col_ner_norm)]
    right_data = data_long[data_long.Metric == col_ner_norm]

    left_chart = alt.Chart(left_data).mark_line(point=True).encode(
        x=alt.X("date:T", title="Year"),
        y=alt.Y("Normalized_Value:Q", title="Left Axis Value"),
        color=alt.Color("Metric:N", title="Metric"),
        tooltip=["date:T", "Metric", "Normalized_Value"]
    )
    
    right_chart = alt.Chart(right_data).mark_line(point=True).encode(
        x=alt.X("date:T"),
        y=alt.Y("Normalized_Value:Q", title="Right Axis Value", axis=alt.Axis(orient="right")),
        color=alt.Color("Metric:N", title="Metric"),
        tooltip=["date:T", "Metric", "Normalized_Value"]
    )
    
    # Layer the charts and let the y-scales be independent
    chart = alt.layer(left_chart, right_chart).resolve_scale(y='independent').properties(
        title=f"{iso3_to_country_name.get(country, country)}: Exchange Rate Metrics (1990-2023)",
        width=600,
        height=400
    ).interactive()
    
    long_term_plots.append(chart)  

long_term_layout = chart_list_to_rows_of_two(long_term_plots)
long_term_layout.display()

In [ ]:
import altair as alt
threshold = ma_5year[t_idx]

df = pd.DataFrame({
    "x": times,
    "y": values,
    "ma": ma_5year,
    "values_integrand": values_integrand,
    "threshold": threshold
})

df_area = pd.DataFrame({
    "x": times[t_idx:idx+1],
    "values_integrand": values_integrand[t_idx:idx+1],
    "threshold": threshold,
})

# --------------------------------------------------------------
# 2.  Build the Altair layers
# --------------------------------------------------------------

base = alt.Chart(df).mark_line(color="#2ca02c").encode(
    x=alt.X("x:T", title="time"),
    y=alt.Y("y:Q", title="R2NER"),
    # y2=alt.Y2("ma:Q", title="5-year moving average"),
)

ma_line = base.mark_line(color="#1f77b4", strokeDash=[5, 5]).encode(
    y=alt.Y("ma:Q", title="5-year moving average")
)

integrand_line = base.mark_line(color="#FF7F0E").encode(
    y=alt.Y("values_integrand:Q", title="Values Integrand (R2NER - 5-year MA)"),
)

threshold_line = base.mark_line(color="#FF0000", strokeDash=[5, 5]).encode(
    y=alt.Y("threshold:Q", title="Threshold (5-year MA * 1.15)"),
)

# Shaded band between the curve and the threshold
area = alt.Chart(df_area).mark_area(opacity=0.25, color="#4682B4").encode(
    x=alt.X("x:T", title="time"),
    y="values_integrand:Q",
    y2=alt.Y2("threshold:Q"))

chart = (base + ma_line + integrand_line + threshold_line + area).properties(
    width=750, height=400,
    title="Integrated area above the threshold"
).interactive()

# chart
points = base.mark_point(filled=True, color="black", size=50, opacity=0).encode(
    tooltip=[
        alt.Tooltip("x:T", title="Date"),
        alt.Tooltip("y:Q", title="R2NER"),
        alt.Tooltip("ma:Q", title="5-year Moving Average"),
        alt.Tooltip("values_integrand:Q", title="Values Integrand")
    ]
)

chart = chart + points
chart
